# 04 — Statistics and EDA

**Package:** `fraud_nb01-07_v2` (flat S3 layout, shared verified helpers)

**Phase 1, notebook 4 of 7.** Exploratory analysis on **train rows only**.

That restriction is not bureaucratic. Anything learned here influences the feature list, the
encoding choices and the model family — so if it is learned from test rows, the test set has
become a selection set and every number reported in notebook 07 is optimistic. The test frame is
carried through untouched and scored exactly once, at the end.

This notebook adds no columns. It re-saves the frame unchanged so the stage chain stays intact.

## 0. Colab bootstrap

Keys come from **Colab Secrets** (key icon, left sidebar) — `AWS_ACCESS_KEY_ID` and
`AWS_SECRET_ACCESS_KEY`, both with notebook access enabled. They are loaded into environment
variables so boto3 still resolves through the default credential chain, keeping the client
construction identical to what runs under IRSA in production. Nothing below prints a key.

In [ ]:
%pip install -q boto3==1.43.95

## 1. Configuration

In [ ]:
# MARKER: fraud_nb01-07_v2 :: 04_Statistics_and_EDA
import io, os, json, time, hashlib, platform, importlib
from datetime import datetime, timezone
import boto3
from botocore.exceptions import ClientError
import joblib
import numpy as np
import pandas as pd
from google.colab import userdata

BUCKET, REGION = "fraud-ecommerce", "ap-south-2"
SPLIT_DATE = pd.Timestamp("2025-07-01")      # stamped in notebook 01 as __split; verified here, never recomputed
CONTRACT_VERSION = "v1"
SEED = 42
MARKER = "fraud_nb01-07_v2"
for _k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    if not os.environ.get(_k):
        os.environ[_k] = userdata.get(_k)     # Colab Secrets -> process env only; never printed or saved
s3 = boto3.client("s3", region_name=REGION)
RAW, LABELS, CONTRACTS, DATA, REPORTS = "raw/", "raw/label_sources/", "contracts/", "data/", "reports/"  # flat layout
ident = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", ident["Account"], "| arn:", ident["Arn"])
if ident["Arn"].endswith(":root"):
    print("NOTE: running as root — accepted for Phase 1; move to an IAM principal before Phase 2")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.isoformat()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o).__name__}")


def read_bytes_s3(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes_s3(body, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"saved s3://{BUCKET}/{key}  ({len(body):,} bytes)")


def key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


def read_s3(key):
    return pd.read_parquet(io.BytesIO(read_bytes_s3(key)))


def save_s3(df, key):
    """Parquet only; the bytes are verified to round-trip columns, dtypes and categories before upload."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    body = buf.getvalue()
    back = pd.read_parquet(io.BytesIO(body))
    assert list(back.columns) == list(df.columns) and len(back) == len(df), f"parquet round-trip changed shape: {key}"
    bad = [c for c in df.columns if str(back[c].dtype) != str(df[c].dtype)]
    assert not bad, f"parquet round-trip changed dtypes in {key}: {bad}"
    badcat = [c for c in df.columns if str(df[c].dtype) == "category"
              and list(back[c].cat.categories) != list(df[c].cat.categories)]
    assert not badcat, f"parquet round-trip changed categories in {key}: {badcat}"
    put_bytes_s3(body, key)


def read_json_s3(key):
    return json.loads(read_bytes_s3(key))


def save_json_s3(obj, key):
    put_bytes_s3(json.dumps(obj, indent=1, default=_jsonable).encode(), key)


def save_model_s3(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    put_bytes_s3(buf.getvalue(), key)


def load_model_s3(key):
    return joblib.load(io.BytesIO(read_bytes_s3(key)))


# names used by notebooks 01-04 (same verified implementations underneath)
def s3_read_csv(key, **kw):
    return pd.read_csv(io.BytesIO(read_bytes_s3(key)), **kw)


s3_read_parquet, s3_read_json = read_s3, read_json_s3


def s3_write_parquet(df, key):
    save_s3(df, key)
    return f"s3://{BUCKET}/{key}"


def s3_write_json(obj, key):
    save_json_s3(obj, key)
    return f"s3://{BUCKET}/{key}"


RAW_KEYS = ([f"{RAW}{t}.csv" for t in ["payments", "orders", "order_items", "account_logins", "customers",
                                         "merchants", "cards", "devices", "ip_reputation"]]
            + [f"{LABELS}{t}.csv" for t in ["fraud_events", "audit_sample", "chargebacks"]]
            + [f"{CONTRACTS}schema_v1.json"])
_absent = [k for k in RAW_KEYS if not key_exists(k)]
assert not _absent, f"missing landing objects in s3://{BUCKET}/: {_absent}"
print(f"landing objects present: {len(RAW_KEYS)} (flat layout, bucket root)")


def run_meta(notebook):
    return {"notebook": notebook, "marker": MARKER,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT}


LIBS = ["pandas", "numpy", "pyarrow", "scipy", "statsmodels", "sklearn", "lightgbm", "xgboost", "joblib", "boto3"]
LIB_VERSIONS = {"python": platform.python_version(),
                **{m: importlib.import_module(m).__version__ for m in LIBS}}
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3", "pyarrow": "23.0.1", "scipy": "1.16.3", "statsmodels": "0.15.0",
            "sklearn": "1.6.1", "lightgbm": "4.6.0", "xgboost": "3.4.1", "boto3": "1.43.95"}
VERSION_DRIFT = {m: {"verified": v, "found": LIB_VERSIONS[m]} for m, v in EXPECTED.items() if LIB_VERSIONS[m] != v}
if not LIB_VERSIONS["python"].startswith("3.13."):
    VERSION_DRIFT["python"] = {"verified": "3.13.x", "found": LIB_VERSIONS["python"]}
print(LIB_VERSIONS)
print("VERSION DRIFT vs the runtime verified on 2026-09-16:", VERSION_DRIFT or "none")

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

## 2. Load and restrict to train

In [ ]:
df = s3_read_parquet(f'{DATA}03_imputed.parquet')
tr = df[df.__split == 'train'].copy()

print(f'full frame : {df.shape[0]:,} x {df.shape[1]}')
print(f'EDA on     : {len(tr):,} TRAIN rows only')
print(f'positives  : {int(tr.is_fraud.sum()):,} ({100*tr.is_fraud.mean():.3f}%)')
print(f'\nlabel sources in train:')
print(tr.groupby('label_source').agg(rows=('payment_id', 'size'),
                                     positives=('is_fraud', 'sum')).to_string())

## 3. Target distribution over time

A fraud rate that trends would mean the base rate itself is drifting, which changes how the gate should be set. A flat rate means volume seasonality and fraud seasonality are separate phenomena.

In [ ]:
m = tr.groupby(tr.payment_ts.dt.to_period('M')).agg(
        payments=('payment_id', 'size'),
        fraud_rate_pct=('is_fraud', lambda s: 100 * s.mean())).round(3)
print(m.to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
m.payments.plot(kind='bar', ax=ax[0], color='#4C78A8')
ax[0].set_title('Monthly payment volume (train)'); ax[0].set_xlabel('')
m.fraud_rate_pct.plot(marker='o', ax=ax[1], color='#E45756')
ax[1].set_title('Observed fraud rate % (train)'); ax[1].set_xlabel('')
plt.tight_layout(); plt.show()

## 4. Amount distribution by class

The reason notebook 03 refused to cap anything.

In [ ]:
print(tr.groupby('is_fraud').payment_amount.describe(
    percentiles=[.25, .5, .75, .95, .99]).round(1).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for lbl, grp in tr.groupby('is_fraud'):
    ax[0].hist(np.log1p(grp.payment_amount), bins=60, alpha=.55, density=True,
               label=f'is_fraud={lbl}')
ax[0].set_title('log1p(payment_amount) by class'); ax[0].legend()
q = tr.groupby(pd.qcut(tr.payment_amount, 20, duplicates='drop'),
               observed=True).is_fraud.mean().mul(100)
ax[1].plot(range(len(q)), q.values, marker='o', color='#E45756')
ax[1].set_title('Fraud rate % by amount ventile'); ax[1].set_xlabel('ventile (low -> high)')
plt.tight_layout(); plt.show()

## 5. Categorical drivers

In [ ]:
for col in ['payment_method', 'shipping_speed', 'merchant_category', 'asn_type',
            'email_domain_class', 'city_tier']:
    g = (tr.groupby(col, observed=True)
           .agg(n=('is_fraud', 'size'), fraud_pct=('is_fraud', lambda s: 100 * s.mean()))
           .sort_values('fraud_pct', ascending=False).round(3))
    g = g[g.n >= 200]
    print(f'--- {col} ---'); print(g.to_string()); print()

## 6. Binary signals, ranked by lift

Lift is the fraud rate when the flag is on, divided by the rate when it is off. Anything near 1.0 carries no information on its own — which does not mean it is useless, only that its contribution is interactive.

In [ ]:
sig = []
for col in ['ip_country_mismatch', 'bill_ship_mismatch', 'is_guest_checkout',
            'is_disposable_email', 'has_coupon', 'is_night', 'is_weekend',
            'ip_country_missing', 'is_retry', 'device_orphan', 'has_card',
            'has_prior_login', 'is_hosting']:
    g = tr.groupby(tr[col].fillna(-1), observed=True)['is_fraud'].agg(['size', 'mean'])
    if 1.0 in g.index and 0.0 in g.index:
        sig.append({'feature': col,
                    'rate_off_pct': round(100 * g.loc[0.0, 'mean'], 3),
                    'rate_on_pct': round(100 * g.loc[1.0, 'mean'], 3),
                    'lift': round(g.loc[1.0, 'mean'] / max(g.loc[0.0, 'mean'], 1e-9), 1),
                    'n_on': int(g.loc[1.0, 'size'])})
sig = pd.DataFrame(sig).sort_values('lift', ascending=False)
print(sig.to_string(index=False))

### `is_retry` is a label artefact, not a signal

Retry rows were assigned fresh `payment_id`s upstream, so they never join to any label source
and are all labelled `0` by construction. Their apparent lift of 0.0 says nothing about fraud —
it says the label join missed them.

A tree will happily learn "retry ⇒ never fraud" from this. The flag is recorded here for
exclusion in notebook 06. This is exactly the class of feature that looks strong in validation
and is worthless in production.

In [ ]:
artefact = tr.groupby('is_retry').agg(rows=('payment_id', 'size'),
                                     positives=('is_fraud', 'sum'))
print(artefact.to_string())
print('\nretry rows joined to a label source:',
      int((tr.is_retry.eq(1) & tr.label_source.ne('unadjudicated')).sum()))

EDA_EXCLUSIONS = {
    'is_retry': 'label-join artefact: retries got fresh payment_ids, so all are labelled 0',
}
print('\nrecorded for exclusion in notebook 06:', EDA_EXCLUSIONS)

## 7. Missingness as signal

`ip_country` is missing more often behind hosting and VPN infrastructure. If the fraud rate differs between missing and present, the indicator earns its place as a feature rather than being imputed away.

In [ ]:
mi = tr.groupby('ip_country_missing').agg(
        rows=('payment_id', 'size'),
        fraud_pct=('is_fraud', lambda s: 100 * s.mean()),
        hosting_pct=('is_hosting', lambda s: 100 * s.mean())).round(3)
print(mi.to_string())
print('\n=> missingness is informative (MAR). The indicator stays as a feature.')

## 8. Numeric correlation with the target

Point-biserial correlation, train rows only. Weak individually is expected at a 0.58% base rate — these are ranked for intuition, not for feature selection.

In [ ]:
num = tr.select_dtypes(include=[np.number]).drop(
    columns=[c for c in ['is_fraud', 'sample_weight'] if c in tr.columns], errors='ignore')
corr = num.corrwith(tr.is_fraud).dropna().sort_values(key=abs, ascending=False)
print(corr.head(20).round(4).to_string())
print('\nweakest:'); print(corr.tail(5).round(4).to_string())

## 9. Save stage 04

No columns added or removed. The frame is re-saved so the stage chain stays unbroken and notebook 05 reads a stage file rather than reaching back two notebooks.

In [ ]:
assert len(df) == df.payment_id.nunique()
print(f'04_eda_complete: {df.shape[0]:,} x {df.shape[1]} (unchanged)')
print(s3_write_parquet(df, f'{DATA}04_eda_complete.parquet'))

record = {**run_meta('04_Statistics_and_EDA'),
          'eda_population': 'train rows only',
          'train_rows': int(len(tr)), 'train_positives': int(tr.is_fraud.sum()),
          'top_lift_features': sig.head(5).to_dict('records'),
          'eda_exclusions': EDA_EXCLUSIONS,
          'missingness_informative': ['ip_country_missing'],
          'columns_added': 0}
print(s3_write_json(record, f'{REPORTS}04_run_record.json'))

## Carried into notebook 05

- **Tests and ranking.** Notebook 05 tests every model candidate on train rows only. Numeric columns get Mann-Whitney U (the amounts are heavily skewed) and categoricals get chi-square. Features are ranked by effect size, with Benjamini-Hochberg-corrected p-values reported but not used to select.
- **Pre-registered hypotheses (Bonferroni α = 0.05/6):** payment method, 3DS success on cards, account age, IP-country mismatch, shared device, and guest checkout.
- **Leakage sentinels, enforced by notebook 06:**
  - near-perfect univariate separation;
  - zero processing fee on non-COD payments;
  - velocity columns not point-in-time.
- **`is_retry` is excluded** from the feature set for the reason recorded above.
- **No production effect.** Test outcomes inform the feature list, never production code.